# VitalDB ICU Admission Prediction

Predicting postoperative ICU admission from preop clinical data, intraop vitals, and waveform features (PPG + ECG).

Holdout split happens first, before any CV or SMOTE. 5-fold CV on the train set only, SMOTE inside folds so we're not leaking synthetic samples across train/val. Bootstrapped CIs on the holdout predictions.

Model order for each signal: signal alone, clinical alone, vitals alone, signal+clinical, clinical+vitals, then everything together. Keeping ECG and PPG on their own native cohorts since extraction success differs slightly between the two signals.

In [ ]:
import pandas as pd
import numpy as np
import vitaldb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from scipy.stats import skew, kurtosis
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

In [ ]:
df = pd.read_csv('clinical_data.csv')
icu_rate = (df['icu_days'] > 0).mean()
print(df.shape)
print(f"icu admit rate: {icu_rate*100:.1f}%")

Feature engineering. ecg_abnormal comes from the preop_ecg text field, anything that isn't literally "Normal Sinus Rhythm" gets flagged.

In [ ]:
data = df.copy()
data['icu_admit'] = (data['icu_days'] > 0).astype(int)
data['ecg_abnormal'] = (data['preop_ecg'] != 'Normal Sinus Rhythm').astype(int)

for c in ['sex', 'department', 'optype']:
    if c in data.columns:
        data[c] = LabelEncoder().fit_transform(data[c].astype(str))

clinical_feats = [
    'age', 'sex', 'bmi', 'asa', 'emop',
    'preop_htn', 'preop_dm', 'ecg_abnormal',
    'preop_hb', 'preop_plt', 'preop_pt', 'preop_aptt',
    'preop_na', 'preop_k', 'preop_gluc', 'preop_alb',
    'preop_cr', 'preop_bun', 'preop_ast', 'preop_alt'
]
vitals_feats = ['hr_mean', 'hr_min', 'hr_max', 'hr_std', 'spo2_mean', 'spo2_min']
ppg_feats = ['ppg_mean', 'ppg_median', 'ppg_std', 'ppg_iqr', 'ppg_range', 'ppg_skew', 'ppg_kurtosis', 'ppg_energy', 'ppg_mad', 'ppg_min', 'ppg_max']
ecg_feats = ['ecg_mean', 'ecg_median', 'ecg_std', 'ecg_iqr', 'ecg_range', 'ecg_skew', 'ecg_kurtosis', 'ecg_energy', 'ecg_mad', 'ecg_min', 'ecg_max']

## Waveform extraction

Pulling stat features off the raw PPG and ECG (Lead II) signal. Tried to get actual clinical intervals (RR/PR/QRS/QT) out of the intraop ECG using neurokit2 first but yield was garbage, like 22-25% of cases, way too low to trust without introducing selection bias. So we're stuck with basic morphology stats instead, which is fine for PPG but part of why ECG underperforms here compared to MIMIC where the intervals are already given.

This part takes a while since it's hitting the vitaldb API per case. Already ran it once, csvs are saved, so the actual extraction is commented out below.

In [ ]:
def get_ppg_feats(caseid):
    try:
        v = vitaldb.load_case(caseid, ['SNUADC/PLETH'])
        sig = v[:, 0]
        sig = sig[~np.isnan(sig)]
        if len(sig) < 1000:
            return None
        lo, hi = np.percentile(sig, [1, 99])
        sig = np.clip(sig, lo, hi).astype(float)
        return {
            'caseid': caseid,
            'ppg_mean': sig.mean(), 'ppg_median': np.median(sig), 'ppg_std': sig.std(),
            'ppg_iqr': np.percentile(sig, 75) - np.percentile(sig, 25),
            'ppg_range': sig.max() - sig.min(),
            'ppg_skew': skew(sig), 'ppg_kurtosis': kurtosis(sig),
            'ppg_energy': np.mean(sig**2), 'ppg_mad': np.mean(np.abs(np.diff(sig))),
            'ppg_min': sig.min(), 'ppg_max': sig.max(),
        }
    except:
        return None

def get_ecg_feats(caseid):
    try:
        v = vitaldb.load_case(caseid, ['SNUADC/ECG_II'])
        sig = v[:, 0]
        sig = sig[~np.isnan(sig)]
        if len(sig) < 1000:
            return None
        lo, hi = np.percentile(sig, [1, 99])
        sig = np.clip(sig, lo, hi).astype(float)
        return {
            'caseid': caseid,
            'ecg_mean': sig.mean(), 'ecg_median': np.median(sig), 'ecg_std': sig.std(),
            'ecg_iqr': np.percentile(sig, 75) - np.percentile(sig, 25),
            'ecg_range': sig.max() - sig.min(),
            'ecg_skew': skew(sig), 'ecg_kurtosis': kurtosis(sig),
            'ecg_energy': np.mean(sig**2), 'ecg_mad': np.mean(np.abs(np.diff(sig))),
            'ecg_min': sig.min(), 'ecg_max': sig.max(),
        }
    except:
        return None

def get_vitals(caseid):
    try:
        v = vitaldb.load_case(caseid, ['Solar8000/HR', 'Solar8000/PLETH_SPO2'])
        hr, spo2 = v[:, 0], v[:, 1]
        hr = hr[~np.isnan(hr)]
        spo2 = spo2[~np.isnan(spo2)]
        row = {'caseid': caseid}
        if len(hr) > 10:
            row.update({'hr_mean': hr.mean(), 'hr_min': hr.min(), 'hr_max': hr.max(), 'hr_std': hr.std()})
        else:
            row.update({'hr_mean': np.nan, 'hr_min': np.nan, 'hr_max': np.nan, 'hr_std': np.nan})
        if len(spo2) > 10:
            row.update({'spo2_mean': spo2.mean(), 'spo2_min': spo2.min()})
        else:
            row.update({'spo2_mean': np.nan, 'spo2_min': np.nan})
        return row
    except:
        return {'caseid': caseid, 'hr_mean': np.nan, 'hr_min': np.nan, 'hr_max': np.nan, 'hr_std': np.nan, 'spo2_mean': np.nan, 'spo2_min': np.nan}

# re-extraction, uncomment if you don't have the csvs already
# ids = data['caseid'].tolist()
# ppg_out, ecg_out, vit_out = [], [], []
# with ThreadPoolExecutor(max_workers=8) as ex:
#     for f in tqdm(as_completed({ex.submit(get_ppg_feats, i): i for i in ids}), total=len(ids)):
#         r = f.result()
#         if r: ppg_out.append(r)
#     for f in tqdm(as_completed({ex.submit(get_ecg_feats, i): i for i in ids}), total=len(ids)):
#         r = f.result()
#         if r: ecg_out.append(r)
#     for f in tqdm(as_completed({ex.submit(get_vitals, i): i for i in ids}), total=len(ids)):
#         vit_out.append(f.result())
# pd.DataFrame(ppg_out).to_csv('vitaldb_ppg_features.csv', index=False)
# pd.DataFrame(ecg_out).to_csv('vitaldb_ecg_features.csv', index=False)
# pd.DataFrame(vit_out).to_csv('vitaldb_hr_spo2.csv', index=False)

ppg_df = pd.read_csv('vitaldb_ppg_features.csv')
ecg_df = pd.read_csv('vitaldb_ecg_features.csv')
vitals_df = pd.read_csv('vitaldb_hr_spo2.csv')
print(len(ppg_df), len(ecg_df), len(vitals_df))

## Helper functions for the modeling loop

In [ ]:
def make_X(df, cols):
    X = df[cols].apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    return X.fillna(X.median())


def fit_with_cv(X, y, name):
    # 5-fold CV with SMOTE applied inside each fold (fit on train fold only, never on val)
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y))
    aucs, prs = [], []
    for tr_i, val_i in kf.split(X, y):
        Xtr, ytr = X.iloc[tr_i], y.iloc[tr_i]
        Xval, yval = X.iloc[val_i], y.iloc[val_i]
        Xtr_s, ytr_s = SMOTE(random_state=SEED).fit_resample(Xtr, ytr)
        clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=0)
        clf.fit(Xtr_s, ytr_s)
        p = clf.predict_proba(Xval)[:, 1]
        oof[val_i] = p
        aucs.append(roc_auc_score(yval, p))
        prs.append(average_precision_score(yval, p))

    cv_auc = roc_auc_score(y, oof)
    cv_pr = average_precision_score(y, oof)
    print(f"{name}: cv auc {cv_auc:.4f} (+/- {np.std(aucs):.4f}), cv pr {cv_pr:.4f} (+/- {np.std(prs):.4f})")

    # pick threshold off the pooled oof preds, not the holdout
    grid = np.linspace(0.01, 0.99, 200)
    f1s = [f1_score(y, (oof >= t).astype(int), zero_division=0) for t in grid]
    thresh = grid[np.argmax(f1s)]

    Xs, ys = SMOTE(random_state=SEED).fit_resample(X, y)
    final_clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=0)
    final_clf.fit(Xs, ys)
    return final_clf, thresh, cv_auc, cv_pr


def boot_ci(y_true, y_pred, fn, n=2000):
    vals = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(fn(y_true[idx], y_pred[idx]))
    return np.mean(vals), np.percentile(vals, 2.5), np.percentile(vals, 97.5)


def boot_delta(y_true, p_a, p_b, n=2000):
    vals = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(roc_auc_score(y_true[idx], p_b[idx]) - roc_auc_score(y_true[idx], p_a[idx]))
    return np.mean(vals), np.percentile(vals, 2.5), np.percentile(vals, 97.5)


def eval_on_holdout(clf, X_test, y_test, thresh, name):
    p = clf.predict_proba(X_test)[:, 1]
    y_arr = np.array(y_test)
    auc_m, auc_lo, auc_hi = boot_ci(y_arr, p, roc_auc_score)
    pr_m, pr_lo, pr_hi = boot_ci(y_arr, p, average_precision_score)
    pred_bin = (p >= thresh).astype(int)
    f1 = f1_score(y_arr, pred_bin, zero_division=0)
    prec = precision_score(y_arr, pred_bin, zero_division=0)
    rec = recall_score(y_arr, pred_bin, zero_division=0)
    print(f"{name} holdout -- auc {auc_m:.4f} ({auc_lo:.4f}-{auc_hi:.4f}), pr {pr_m:.4f} ({pr_lo:.4f}-{pr_hi:.4f}), f1 {f1:.4f}, prec {prec:.4f}, rec {rec:.4f}")
    return p, {'roc': auc_m, 'roc_ci': (auc_lo, auc_hi), 'pr': pr_m, 'pr_ci': (pr_lo, pr_hi), 'f1': f1, 'precision': prec, 'recall': rec, 'threshold': thresh}

## ECG models

signal -> clinical -> vitals -> signal+clinical -> clinical+vitals -> everything

In [ ]:
ecg_merged = data.merge(vitals_df, on='caseid').merge(ecg_df, on='caseid')
print(f"ecg cohort: {len(ecg_merged)}, icu+ rate {ecg_merged['icu_admit'].mean()*100:.1f}%")

ecg_train, ecg_test = train_test_split(ecg_merged, test_size=0.2, stratify=ecg_merged['icu_admit'], random_state=SEED)
ecg_train = ecg_train.reset_index(drop=True)
ecg_test = ecg_test.reset_index(drop=True)
y_tr_ecg = ecg_train['icu_admit'].astype(int)
y_te_ecg = ecg_test['icu_admit'].astype(int)

ecg_runs = {}
specs_ecg = [
    ('ecg', ecg_feats),
    ('clinical', clinical_feats),
    ('vitals', vitals_feats),
    ('ecg_clinical', ecg_feats + clinical_feats),
    ('clinical_vitals', clinical_feats + vitals_feats),
    ('full', ecg_feats + clinical_feats + vitals_feats),
]

for name, feats in specs_ecg:
    Xtr = make_X(ecg_train, feats)
    Xte = make_X(ecg_test, feats)
    clf, thresh, cv_auc, cv_pr = fit_with_cv(Xtr, y_tr_ecg, name)
    preds, metrics = eval_on_holdout(clf, Xte, y_te_ecg, thresh, name)
    metrics['cv_roc'] = cv_auc
    metrics['cv_pr'] = cv_pr
    ecg_runs[name] = {'preds': preds, 'metrics': metrics}

## PPG models, same structure

In [ ]:
ppg_merged = data.merge(vitals_df, on='caseid').merge(ppg_df, on='caseid')
print(f"ppg cohort: {len(ppg_merged)}, icu+ rate {ppg_merged['icu_admit'].mean()*100:.1f}%")

ppg_train, ppg_test = train_test_split(ppg_merged, test_size=0.2, stratify=ppg_merged['icu_admit'], random_state=SEED)
ppg_train = ppg_train.reset_index(drop=True)
ppg_test = ppg_test.reset_index(drop=True)
y_tr_ppg = ppg_train['icu_admit'].astype(int)
y_te_ppg = ppg_test['icu_admit'].astype(int)

ppg_runs = {}
specs_ppg = [
    ('ppg', ppg_feats),
    ('clinical', clinical_feats),
    ('vitals', vitals_feats),
    ('ppg_clinical', ppg_feats + clinical_feats),
    ('clinical_vitals', clinical_feats + vitals_feats),
    ('full', ppg_feats + clinical_feats + vitals_feats),
]

for name, feats in specs_ppg:
    Xtr = make_X(ppg_train, feats)
    Xte = make_X(ppg_test, feats)
    clf, thresh, cv_auc, cv_pr = fit_with_cv(Xtr, y_tr_ppg, name)
    preds, metrics = eval_on_holdout(clf, Xte, y_te_ppg, thresh, name)
    metrics['cv_roc'] = cv_auc
    metrics['cv_pr'] = cv_pr
    ppg_runs[name] = {'preds': preds, 'metrics': metrics}

## How much does each signal actually add

In [ ]:
def show_deltas(runs, y_test, sig):
    y_arr = np.array(y_test)
    a = boot_delta(y_arr, runs['clinical']['preds'], runs[f'{sig}_clinical']['preds'])
    b = boot_delta(y_arr, runs[sig]['preds'], runs[f'{sig}_clinical']['preds'])
    c = boot_delta(y_arr, runs['clinical_vitals']['preds'], runs[f'{sig}_clinical']['preds'])
    d = boot_delta(y_arr, runs[f'{sig}_clinical']['preds'], runs['full']['preds'])
    e = boot_delta(y_arr, runs['clinical_vitals']['preds'], runs['full']['preds'])
    print(f"\n{sig}:")
    print(f"  signal+clin vs clin alone:      {a[0]:+.4f} ({a[1]:+.4f}, {a[2]:+.4f})")
    print(f"  signal+clin vs signal alone:    {b[0]:+.4f} ({b[1]:+.4f}, {b[2]:+.4f})")
    print(f"  signal+clin vs vitals+clin:     {c[0]:+.4f} ({c[1]:+.4f}, {c[2]:+.4f})")
    print(f"  full vs signal+clin:            {d[0]:+.4f} ({d[1]:+.4f}, {d[2]:+.4f})")
    print(f"  full vs clin+vitals:            {e[0]:+.4f} ({e[1]:+.4f}, {e[2]:+.4f})")

show_deltas(ecg_runs, y_te_ecg, 'ecg')
show_deltas(ppg_runs, y_te_ppg, 'ppg')

In [ ]:
out = {
    'ecg': {k: v['metrics'] for k, v in ecg_runs.items()},
    'ppg': {k: v['metrics'] for k, v in ppg_runs.items()},
}
with open('vitaldb_icu_admission_results.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)

## Plots

In [ ]:
mpl.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False, 'figure.dpi': 300})

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
colors = {'full': '#2ECC71', 'clinical_vitals': '#D65F5F'}

for col, (runs, ytest, sig) in enumerate([(ecg_runs, y_te_ecg, 'ecg'), (ppg_runs, y_te_ppg, 'ppg')]):
    ax = axes[0, col]
    for name, c in [('full', colors['full']), ('clinical_vitals', colors['clinical_vitals']), (f'{sig}_clinical', '#4878CF')]:
        p = runs[name]['preds']
        auc = runs[name]['metrics']['roc']
        fpr, tpr, _ = roc_curve(ytest, p)
        ax.plot(fpr, tpr, lw=2, color=c, label=f'{name} ({auc:.3f})')
    ax.plot([0,1],[0,1],'--',color='gray',lw=1)
    ax.set_xlabel('1 - specificity'); ax.set_ylabel('sensitivity')
    ax.set_title(f'{sig.upper()} ROC')
    ax.legend(fontsize=8)

    ax2 = axes[1, col]
    for name, c in [('full', colors['full']), ('clinical_vitals', colors['clinical_vitals']), (f'{sig}_clinical', '#4878CF')]:
        p = runs[name]['preds']
        ap = runs[name]['metrics']['pr']
        prec, rec, _ = precision_recall_curve(ytest, p)
        ax2.plot(rec, prec, lw=2, color=c, label=f'{name} ({ap:.3f})')
    ax2.set_xlabel('recall'); ax2.set_ylabel('precision')
    ax2.set_title(f'{sig.upper()} PR')
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('vitaldb_icu_admission_roc_pr.png', dpi=300, bbox_inches='tight')
plt.show()